# 多输入多输出通道

实现一下多输入通道的互相关运算

In [1]:
import torch
from torch import nn
from d2l import torch as d2l

def multi_crossrelated(X, K):
    return sum([d2l.corr2d(x, k) for x, k in zip(X, K)])

X = torch.tensor([[[0, 1, 2], [3, 4, 5], [6, 7, 8]], 
                [[1, 2, 3], [4, 5, 6], [7, 8, 9]]])
K = torch.tensor([[[0, 1], [2, 3]], [[1, 2], [3, 4]]])
multi_crossrelated(X, K)


tensor([[ 56.,  72.],
        [104., 120.]])

计算多个通道输出的互相关函数

In [2]:
def multio_crossrelated(X, K):
    return torch.stack([multi_crossrelated(X, k) for k in K], 0)
K = torch.stack((K, K + 1, K + 2), 0)
Y = multio_crossrelated(X, K)
Y

tensor([[[ 56.,  72.],
         [104., 120.]],

        [[ 76., 100.],
         [148., 172.]],

        [[ 96., 128.],
         [192., 224.]]])

1×1卷积

In [3]:
def crossrelated_1x1(X, K):
    c_in, nh, nw = X.shape
    c_o = K.shape[0]
    K = K.reshape(c_o, c_in)
    X = X.reshape(c_in, nh*nw)
    Y = torch.matmul(K, X)
    Y = Y.reshape(c_o, nh, nw)
    return Y

X = torch.normal(0, 1, (3, 3, 3))
K = torch.normal(0, 1, (2, 3, 1, 1))
Y1 = multio_crossrelated(X, K)
Y2 = crossrelated_1x1(X, K)
assert torch.abs(Y1 - Y2).sum() < 1e-6 